# SHG mode survey: driver notebook

This notebook runs the scripts in this folder (`survey.py`, `geometry.py`, `summarize_run.py`,
`plot_crossing.py`, `cerenkov_run.py`). All the physics lives in those `.py` files; the notebook only
calls them and displays the results. Background: [PLAN.md](PLAN.md); command-line equivalents: [README.md](README.md).

**Current task: loaded-family smoke test** (PLAN.md open item 1):

| Step | What | EMode? | Time |
|---|---|---|---|
| A | Materials check: is `TiO2` in EMode's database, and what n, k does it give at the pump and SH wavelengths? | yes | ~10 s |
| B | Geometry check: build 3 loaded variants, compare EMode's index plot and pump modes with the masks `geometry.py` uses for the physics (verifies the partial-etch and `strip_offset` conventions) | yes | ~1–2 min |
| C | Smoke survey: one loaded geometry through the full Step 1 survey | yes | ~5–10 min |
| D | Results: tables, summary, dispersion and field plots | no | seconds |
| E | Cerenkov (Step 1b) for the smoke run's pumps | no | ~1 min/pump |
| F | Full 8-geometry loaded sweep, launched **detached** | yes | ~45–90 min |

Before you start:
* **EMode is single-seat.** Close any EMode session on the laptop first.
* **Kernel:** UVPY_venv.
* **Don't edit or save this `.ipynb` from outside VS Code while a cell is running.** If Claude changes the file on disk,
  run **File: Revert File** before you run anything, or VS Code will overwrite the changes.
* Steps A–C start EMode with `clear='all'`, which **kills any other EMode session on this machine**.

In [ ]:
# --- Setup (run first; rerun after editing any .py in this folder -- it reloads them) ---
import glob
import importlib
import json
import os
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

HERE = os.path.abspath('')                      # the notebook's folder = shg_mode_survey/
assert os.path.exists(os.path.join(HERE, 'survey.py')), f"open this notebook from shg_mode_survey/ (cwd={HERE})"
sys.path.insert(0, HERE)
import geometry as geo
import survey
importlib.reload(geo); importlib.reload(survey)
pmh = survey.pmh                                # emode_helpers (launch_session etc.)
PY = sys.executable                             # the kernel's interpreter, used for the subprocess steps


def run_script(*args):
    # Run one of this folder's scripts in a subprocess (keeps their matplotlib 'Agg' backend out of the kernel).
    r = subprocess.run([PY, '-u', *args], cwd=HERE, capture_output=True, text=True)
    print(r.stdout[-4000:])
    if r.returncode:
        print(r.stderr[-4000:])
        raise RuntimeError(f"{args[0]} exited with {r.returncode}")


print('python:', PY)

## A. Materials check

The loaded configs use `strip_material='TiO2'` from EMode's built-in database. TiO2 absorbs strongly
at 215–235 nm, so the SH loss depends on the k value EMode uses. **Check that the SH-band k is
not zero.** If `TiO2` is missing, or its UV data looks wrong, set `strip_eq` (an n + k·j equation in µm) in the config instead.
`alpha` is the bulk power absorption 4πk/λ.

In [ ]:
MATERIALS = ['TiO2', 'Al2O3', 'SiO2']
WAVELENGTHS = [215, 225, 235, 430, 450, 470]      # nm: SH band, pump band

em = pmh.launch_session('shg_material_check', clear='all')
rows = []
try:
    for mat in MATERIALS:
        for wl in WAVELENGTHS:
            try:
                raw = em.refractive_index(material=mat, wavelength=wl)
                v = np.atleast_1d(np.asarray(raw, dtype=complex))
                n, k = float(np.real(v).max()), float(np.abs(np.imag(v)).max())
                rows.append({'material': mat, 'wavelength_nm': wl, 'n': n, 'k': k,
                             'alpha_dB_per_um': 10 * np.log10(np.e) * 4 * np.pi * k / (wl * 1e-3),
                             'raw': str(raw)})
            except Exception as e:  # noqa: BLE001
                rows.append({'material': mat, 'wavelength_nm': wl, 'raw': f'ERROR: {e!r}'})
finally:
    em.close(save=False)
mat_df = pd.DataFrame(rows)
display(mat_df.style.format({'n': '{:.4f}', 'k': '{:.4g}', 'alpha_dB_per_um': '{:.3g}'}, na_rep='-'))

## B. Geometry convention check

`geometry.Loaded` builds the structure in EMode **and**, separately, draws numpy masks of the same
regions. The masks are used for the χ(2) map, the absorption regions and the Cerenkov ε(x, y). The
two have to agree. Two EMode conventions in the loaded family have never been checked:

* **partial etch**: `mask` = width of the unetched rib, and a slab of `t_film − etch_depth` is left outside it;
* **`mask_offset`**: shifts the strip laterally, and in which direction (the masks assume +x).

For each test case the next cell builds the structure, solves a few modes at the pump wavelength, and shows:
1. EMode's own index plot (saved as a PNG);
2. the `geometry.py` masks next to the lowest modes' |E|², with the mask outlines drawn on top.

**What to look for:**
* **smoke**: an unetched film, a 70 × 500 nm strip centred on top, and the lowest TM-like mode under the strip.
* **partial_etch**: an 800 nm-wide rib, 354 nm tall, with a 204 nm slab outside it. The EMode PNG must match the mask panel.
* **offset**: the strip, and the mode's x-centroid, at **+300 nm**. If EMode puts it at −300, the
  sign convention is flipped and `Loaded._masks` has to change.
  The asymmetric case solves with `boundary_condition='00'` (no mirror), the same as the survey does.

The EMode outlines and the mask outlines must line up to within a grid cell (10 nm).

In [ ]:
CHECK_WAVELENGTH = 450.0        # nm (pump band centre)
CHECK_NUM_MODES = 4
CHECK_RESOLUTION = 10.0         # nm

BASE = {'t_film': 354.0, 'etch_depth': 0.0, 'strip_material': 'TiO2', 'h_s': 70.0, 'w_s': 500.0}
CHECK_CASES = {
    'smoke':        ('loaded', dict(BASE)),
    'partial_etch': ('loaded', {**BASE, 'etch_depth': 150.0, 'w_rib': 800.0}),
    'offset':       ('loaded', {**BASE, 'strip_offset': 300.0}),
}
CHECK_DIR = os.path.join(HERE, 'runs', 'geometry_check')
os.makedirs(CHECK_DIR, exist_ok=True)


def region_masks(g, x, y):
    # {name: bool mask (ny, nx)} as geometry.py sees it
    if hasattr(g, '_masks'):
        film, strip = g._masks(x, y)
        return {'film': film, 'strip': strip}
    return {'core': g.core_mask(x, y)}


def check_geometry(name, family, params):
    g = geo.make_geometry(family, params)
    bc = survey.sh_bc(g)                     # '0A' if mirror-symmetric, else '00'
    png_base = os.path.join(CHECK_DIR, f'{name}_index')   # absolute path: em.plot needs it
    for old in glob.glob(png_base + '*'):
        os.remove(old)
    em = pmh.launch_session(f'shg_check_{name}', clear='all')
    t0 = time.time()
    try:
        g.build(em)
        em.settings(boundary_condition=bc, wavelength=CHECK_WAVELENGTH, num_modes=CHECK_NUM_MODES,
                    x_resolution=CHECK_RESOLUTION, y_resolution=CHECK_RESOLUTION)
        n, te = survey.solve(em, f'check_{name}')
        x, y, modes = survey.fetch_fields(em, survey.SCAN_KEYS)
        plot_note = ''
        for comp in ('Index', 'Shapes'):
            try:
                em.plot(component=comp, file_name=png_base, file_type='png')
                plot_note = comp
                break
            except Exception as e:  # noqa: BLE001
                plot_note += f'em.plot({comp!r}) failed: {e!r}  '
    finally:
        em.close(save=False)
    print(f"=== {name}: {g.describe()}  bc={bc}  ({time.time() - t0:.0f}s)")
    pngs = glob.glob(png_base + '*')
    if pngs:
        print(f"EMode {plot_note} plot: {pngs[0]}")
        display(Image(filename=pngs[0], width=700))
    else:
        print('no EMode plot:', plot_note)

    masks = region_masks(g, x, y)
    x0, x1, y0, y1 = g.bbox()
    pad = 500
    xs, ys = (x >= x0 - pad) & (x <= x1 + pad), (y >= y0 - pad) & (y <= y1 + pad)
    ext = [x[xs][0], x[xs][-1], y[ys][0], y[ys][-1]]
    crop = lambda a: a[np.ix_(ys, xs)]
    region_img = np.zeros((len(y), len(x)))
    region_img[y[:, None] < g.params['substrate_height'] + 0 * x[None, :]] = 1
    for i, m in enumerate(masks.values(), start=2):
        region_img[m] = i
    colors = ['C0', 'C1', 'C3', 'C2']

    fig, axs = plt.subplots(1, 1 + len(modes), figsize=(4.2 * (1 + len(modes)), 3.8))
    axs[0].imshow(crop(region_img), origin='lower', extent=ext, cmap='Greys', aspect='equal')
    axs[0].set_title('geometry.py masks\n(sub / ' + ' / '.join(masks) + ')', fontsize=9)
    for k, f in enumerate(modes):
        I = np.abs(f['Ex'])**2 + np.abs(f['Ey'])**2 + np.abs(f['Ez'])**2
        xc = float(np.sum(I * x[None, :]) / np.sum(I))
        ax = axs[k + 1]
        ax.imshow(crop(I), origin='lower', extent=ext, cmap='magma', aspect='equal')
        ax.set_title(f"mode {k}: n_eff={n[k].real:.4f}\nTE frac={te[k]:.2f}  x-centroid={xc:+.0f} nm", fontsize=9)
        print(f"  mode {k}: n_eff={n[k].real:.5f}  TE fraction={te[k]:.2f}  |E|^2 x-centroid={xc:+.1f} nm")
    for ax in axs:
        for (mname, m), c in zip(masks.items(), colors):
            ax.contour(x[xs], y[ys], crop(m).astype(float), levels=[0.5], colors=c, linewidths=0.9)
        ax.axhline(g.params['substrate_height'], color='C0', lw=0.6, ls='--')
        ax.set_xlabel('x [nm]')
    fig.tight_layout()
    plt.show()
    off = g.params.get('strip_offset', 0.0)
    if off:
        print(f"  strip_offset = {off:+g} nm -> mask strip centred at x = {off:+g}; compare with the EMode plot "
              f"and the TM-like mode's x-centroid above")


for name, (family, params) in CHECK_CASES.items():
    check_geometry(name, family, params)

**If a convention is wrong:** stop here and tell Claude what you saw (or paste the plots). The fix goes in
`geometry.Loaded._masks` (or `build`), and nothing downstream should be trusted until it's made.

## C. Smoke survey (one geometry, full Step 1)

This runs `survey.main()` inside the kernel, so the progress lines print below. If the kernel is
interrupted or restarted, rerun the cell: the run resumes, and a geometry that finished (status `ok`) is
skipped. To force a fresh run, change `RUN_NAME`.

`SMOKE` is defined in `survey_config_loaded.py`: t_film = 354, unetched, TiO2 strip h_s = 70, w_s = 500.

In [ ]:
RUN_NAME = 'loaded_smoke'
CONFIG = 'survey_config_loaded_smoke'

for m in ('survey_config', 'survey_config_loaded', CONFIG):   # pick up any edits to the configs
    importlib.reload(importlib.import_module(m))
importlib.reload(geo); importlib.reload(survey)
t0 = time.time()
survey.main(RUN_NAME, CONFIG)
print(f"\nfinished in {(time.time() - t0) / 60:.1f} min -> runs/{RUN_NAME}/")

## D. Results

* `geometries.csv`: status (`ok`/`failed`) and the error, if any.
* `pump_modes.csv`: the pump modes found, with labels, n_eff, losses and Cerenkov flags.
  For a loaded guide, check that the lowest pumps are strip-confined TM modes and not slab modes.
* `crossings.csv`: the phase matches. NCE is lossless; `peak` is the loss-limited efficiency, and SH
  loss through the absorbing TiO2 counts here.

In [ ]:
RUN_DIR = os.path.join(HERE, 'runs', RUN_NAME)
geoms = pd.read_csv(os.path.join(RUN_DIR, 'geometries.csv'))
display(geoms[['fingerprint', 'family', 'status', 'n_pump_modes', 'n_crossings', 'n_refined', 'seconds', 'error']])

pumps = pd.read_csv(os.path.join(RUN_DIR, 'pump_modes.csv'))
display(pumps[['fingerprint', 'pump_id', 'label', 'sym_class', 'te_fraction', 'n_eff_center',
               'scattering_dB_per_m', 'absorption_dB_per_m', 'A_eff_um2',
               'cerenkov_allowed_any', 'cerenkov_best_angle_deg']])

cross = pd.read_csv(os.path.join(RUN_DIR, 'crossings.csv'))
cols = [c for c in ['pump_label', 'sh_label', 'sh_te_fraction', 'wavelength_sh', 'eta_pct_per_W_cm2',
                    'overlap_shape', 'A_eff_pump_um2', 'A_eff_sh_um2', 'pump_scattering_dB_per_m',
                    'sh_scattering_dB_per_m', 'sh_absorption_dB_per_m', 'L_opt_mm',
                    'peak_efficiency_pct_per_W', 'status'] if c in cross.columns]
print(f"{len(cross)} crossings, {(cross['status'] == 'refined').sum()} refined")
display(cross.sort_values('eta_pct_per_W_cm2', ascending=False)[cols].head(20))

In [ ]:
# Summary markdown + dispersion diagram(s) + field plots of the top refined crossings
run_script('summarize_run.py', RUN_NAME)
run_script('plot_crossing.py', RUN_NAME, '8')
display(Markdown(open(os.path.join(RUN_DIR, 'summary', 'summary.md'), encoding='utf-8').read()))
for p in sorted(glob.glob(os.path.join(RUN_DIR, 'summary', 'dispersion', '*.png'))):
    display(Image(filename=p, width=650))
for p in sorted(glob.glob(os.path.join(RUN_DIR, 'plots', '*.png'))):
    display(Image(filename=p))

## E. Cerenkov / radiated SH (Step 1b): optional

This step needs no EMode. It runs the driven FDFD solve on each exported pump mode. With `ALL_PUMPS=False`,
only the pumps flagged Cerenkov-allowed are solved. The unetched AlN slab is a *finite* Cerenkov region
(the SH couples into slab modes), so `to_left`/`to_right` show how much of the radiation goes sideways into the slab.

In [ ]:
ALL_PUMPS = False
run_script('cerenkov_run.py', RUN_NAME, *(['--all'] if ALL_PUMPS else []))
cpath = os.path.join(RUN_DIR, 'cerenkov.csv')
if os.path.exists(cpath):
    display(pd.read_csv(cpath)[['label', 'n_eff', 'wavelength_sh', 'cerenkov_allowed_any', 'kappa_pct_per_W_cm',
                                'to_substrate', 'to_topclad', 'to_left', 'to_right', 'balance', 'seconds']])

## F. Full loaded sweep (detached): only after B and C look right

This launches `survey.py` as a separate background process, so the run survives closing VS Code or restarting the
kernel. The geometries are the 8 cases in `GEOMETRIES` in `survey_config_loaded.py`; edit that file to change them.
Use the monitor cell to check progress. Rerunning the launch cell after a crash resumes the run.

Don't launch this while another EMode run (or Steps A–C) is active: the license is single-seat, and the
`clear='all'` in each run would kill the other session.

In [ ]:
SWEEP_RUN = 'loaded_sweep1'
SWEEP_CONFIG = 'survey_config_loaded'


def launch_detached(run_name, config_module):
    out = open(os.path.join(HERE, f'survey_stdout_{run_name}.log'), 'a')
    err = open(os.path.join(HERE, f'survey_stdout_{run_name}.err.log'), 'a')
    flags = subprocess.DETACHED_PROCESS | subprocess.CREATE_NEW_PROCESS_GROUP
    args = [PY, '-u', 'survey.py', run_name, config_module]
    try:   # break out of the kernel's job object, if it has one, so a kernel restart doesn't kill the run
        p = subprocess.Popen(args, cwd=HERE, stdout=out, stderr=err, stdin=subprocess.DEVNULL,
                             creationflags=flags | subprocess.CREATE_BREAKAWAY_FROM_JOB)
    except OSError:
        p = subprocess.Popen(args, cwd=HERE, stdout=out, stderr=err, stdin=subprocess.DEVNULL,
                             creationflags=flags)
    print(f"launched PID {p.pid}: runs/{run_name}/progress.log")
    return p


sweep_proc = launch_detached(SWEEP_RUN, SWEEP_CONFIG)

In [ ]:
# Monitor: rerun any time (doesn't touch EMode)
plog = os.path.join(HERE, 'runs', SWEEP_RUN, 'progress.log')
print(''.join(open(plog, encoding='utf-8').readlines()[-25:]) if os.path.exists(plog) else 'no progress.log yet')
errlog = os.path.join(HERE, f'survey_stdout_{SWEEP_RUN}.err.log')
if os.path.exists(errlog) and os.path.getsize(errlog):
    print('--- stderr (tail) ---\n' + open(errlog, encoding='utf-8', errors='replace').read()[-2000:])

When the sweep finishes, set `RUN_NAME = SWEEP_RUN` and rerun the Step D (and E) cells to see its results.